In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
import os
Neo4j_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

In [3]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(url=Neo4j_URI,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)

In [4]:
graph

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b")


In [6]:
#llm.invoke('Tell me a joke about 2 lines on famout or hot topics these days')

In [7]:
from langchain_core.documents import Document


In [8]:
text = """
Elon Musk is a South African–born American entrepreneur who cofounded the electronic payment firm PayPal and formed SpaceX, maker of launch vehicles and spacecraft.
He was also one of the first significant investors in, as well as chief executive officer of, the electric car manufacturer Tesla.
In addition, Musk acquired Twitter (later X) in 2022.
Musk was born June 28, 1971, in Pretoria, South Africa, to a South African father and a Canadian mother.
He displayed an early talent for computers and entrepreneurship, and at age 12, created a video game that he sold to a computer magazine.
"""

In [9]:
documents = [Document(page_content=text)]
documents

[Document(metadata={}, page_content='\nElon Musk is a South African–born American entrepreneur who cofounded the electronic payment firm PayPal and formed SpaceX, maker of launch vehicles and spacecraft.\nHe was also one of the first significant investors in, as well as chief executive officer of, the electric car manufacturer Tesla.\nIn addition, Musk acquired Twitter (later X) in 2022.\nMusk was born June 28, 1971, in Pretoria, South Africa, to a South African father and a Canadian mother.\nHe displayed an early talent for computers and entrepreneurship, and at age 12, created a video game that he sold to a computer magazine.\n')]

In [10]:
from langchain_experimental.graph_transformers import LLMGraphTransformer


In [11]:
llm_transformer = LLMGraphTransformer(llm=llm)

In [ ]:
#graph_docs = llm_transformer.convert_to_graph_documents(documents=documents)

BadRequestError: Error code: 400 - {'error': {'message': 'Failed to parse tool call arguments as JSON', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "DynamicGraph", "arguments": {"nodes":[{"id":"Elon Musk","type":"Person"},{"id":"PayPal","type":"Company"},{"id":"SpaceX","type":"Company"},{"id":"Tesla","type":"Company"},{"id":"Twitter","type":"Company"},{"id":"South Africa","type":"Country"},{"id":"United States","type":"Country"},{"id":"Pretoria","type":"City"},{"id":"South African father","type":"Person"},{"id":"Canadian mother","type":"Person"},{"id":"Video game","type":"Product"},{"id":"Computer magazine","type":"Publication"}],"relationships":[{"source_node_id":"Elon Musk","source_node_type":"Person","target_node_id":"PayPal","target_node_type":"Company","type":"COFOUNDED"},{"source_node_id":"Elon Musk","source_node_type":"Person","target_node_id":"SpaceX","target_node_type":"Company","type":"FORMED"},{"source_node_id":"Elon Musk","sour

In [ ]:
#graph_docs

[GraphDocument(nodes=[Node(id='Elon Musk', type='Person', properties={}), Node(id='Paypal', type='Company', properties={}), Node(id='Spacex', type='Company', properties={}), Node(id='Tesla', type='Company', properties={}), Node(id='Twitter', type='Company', properties={}), Node(id='X', type='Company', properties={}), Node(id='South Africa', type='Country', properties={}), Node(id='Canada', type='Country', properties={})], relationships=[Relationship(source=Node(id='Elon Musk', type='Person', properties={}), target=Node(id='Paypal', type='Company', properties={}), type='COFOUNDER', properties={}), Relationship(source=Node(id='Elon Musk', type='Person', properties={}), target=Node(id='Spacex', type='Company', properties={}), type='FOUNDER', properties={}), Relationship(source=Node(id='Elon Musk', type='Person', properties={}), target=Node(id='Tesla', type='Company', properties={}), type='CEO', properties={}), Relationship(source=Node(id='Elon Musk', type='Person', properties={}), target=

In [ ]:
# graph_docs[0].nodes

[Node(id='Elon Musk', type='Person', properties={}),
 Node(id='Paypal', type='Company', properties={}),
 Node(id='Spacex', type='Company', properties={}),
 Node(id='Tesla', type='Company', properties={}),
 Node(id='Twitter', type='Company', properties={}),
 Node(id='X', type='Company', properties={}),
 Node(id='South Africa', type='Country', properties={}),
 Node(id='Canada', type='Country', properties={})]

In [ ]:
movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In [ ]:
# CSV data exectued in neo4j
graph.query(movie_query)

[]

In [13]:
# Printing Graph Schema
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [14]:
from langchain_neo4j import GraphCypherQAChain


In [15]:
chain = GraphCypherQAChain.from_llm(llm=llm,graph=graph,allow_dangerous_requests=True,verbose=True)

In [16]:
chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x0000013F2F4D6ED0>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, '

In [19]:
response=chain.invoke({"query":"tell me the director of Sabrina"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Sabrina"})
RETURN p.name AS director;
Full Context:
[{'director': 'Sydney Pollack'}]

> Finished chain.


{'query': 'tell me the director of Sabrina', 'result': 'Sydney Pollack.'}

In [20]:
response=chain.invoke({"query":"Who were the actors of the movie Four Rooms"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m:Movie {title: "Four Rooms"})
RETURN p.name;
Full Context:
[{'p.name': 'Amanda De Cadenet'}, {'p.name': 'Valeria Golino'}, {'p.name': 'Madonna'}, {'p.name': 'Sammi Davis'}]

> Finished chain.


{'query': 'Who were the actors of the movie Four Rooms',
 'result': 'Amanda De Cadenet, Valeria Golino, Madonna, and Sammi Davis were the actors of the movie *Four Rooms*.'}

In [22]:
response=chain.invoke({"query":"Who were the actors and directors of the movie Four Rooms"})

response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: "Four Rooms"})
OPTIONAL MATCH (d:Person)-[:DIRECTED]->(m)
OPTIONAL MATCH (a:Person)-[:ACTED_IN]->(m)
RETURN collect(d.name) AS directors, collect(a.name) AS actors;
Full Context:
[{'directors': ['Allison Anders', 'Allison Anders', 'Allison Anders', 'Allison Anders', 'Robert Rodriguez', 'Robert Rodriguez', 'Robert Rodriguez', 'Robert Rodriguez', 'Quentin Tarantino', 'Quentin Tarantino', 'Quentin Tarantino', 'Quentin Tarantino', 'Alexandre Rockwell', 'Alexandre Rockwell', 'Alexandre Rockwell', 'Alexandre Rockwell'], 'actors': ['Amanda De Cadenet', 'Valeria Golino', 'Madonna', 'Sammi Davis', 'Amanda De Cadenet', 'Valeria Golino', 'Madonna', 'Sammi Davis', 'Amanda De Cadenet', 'Valeria Golino', 'Madonna', 'Sammi Davis', 'Amanda De Cadenet', 'Valeria Golino', 'Madonna', 'Sammi Davis']}]

> Finished chain.


{'query': 'Who were the actors and directors of the movie Four Rooms',
 'result': 'The directors were Allison Anders, Robert Rodriguez, Quentin Tarantino, and Alexandre\u202fRockwell. The actors were Amanda\u202fDe\u202fCadenet, Valeria\u202fGolino, Madonna, and Sammi\u202fDavis.'}